# Chapter 4 Project 1 - Constrained Attitude Control

A double-integrator attitude model is used as the first application study. The point is to compare LQR, saturated LQR, constrained MPC, rate limits, terminal-cost tuning, and soft constraint recovery while keeping every MPC ingredient visible.

In [ ]:
%matplotlib inline
import os
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np

from systems.double_integrator import attitude_matrices
from controllers.lqr import discrete_lqr
from mpc.casadi_linear_mpc import solve_linear_mpc
from scenarios.ch4_project1_attitude import run_project1

cwd = Path.cwd()
repo_root = cwd if (cwd / "scenarios").exists() else cwd.parent
output_root = Path(os.environ.get("THIMPC_OUTPUT_ROOT", repo_root / "outputs"))

dt = 0.1
A, B = attitude_matrices(dt=dt)
print("A =")
print(A)
print("B =")
print(B)


## Instructor solution

The marked block contains one reasonable starting design. In the public notebook this becomes a TODO region, while the baseline values keep the notebook executable.

In [ ]:
# Baseline values keep the public notebook runnable.
Q = np.diag([5.0, 1.0])
R = np.array([[1.0]])
horizon = 10
u_bounds = (np.array([-1.1]), np.array([1.1]))
x_bounds = (np.array([-1.2, -2.5]), np.array([1.2, 2.5]))
rate_bound = np.array([0.25])
soften_theta_constraint = False
slack_penalty = 1000.0

# SOLUTION_START
Q = np.diag([20.0, 2.0])
R = np.array([[0.25]])
horizon = 18
soften_theta_constraint = True
slack_penalty = 5000.0
# SOLUTION_END

K, P_infty = discrete_lqr(A, B, Q, R)
print("Q =")
print(Q)
print("R =")
print(R)
print("P_infty =")
print(P_infty)
print("input bounds:", u_bounds)
print("state bounds:", x_bounds)
print("rate bound:", rate_bound)
print("soft theta constraint:", soften_theta_constraint)


## Visible MPC Solve

Here the dynamics, stage cost, terminal cost, hard bounds, rate bound, optional slack variable, slack penalty, solver status, and first applied input are passed or inspected explicitly.

In [ ]:
x0 = np.array([1.0, 0.0])
result = solve_linear_mpc(
    A,
    B,
    x0,
    horizon,
    Q,
    R,
    P_terminal=P_infty,
    u_bounds=u_bounds,
    x_bounds=x_bounds,
    rate_bound=rate_bound,
    u_previous=np.array([0.0]),
    soften_state_indices=[0] if soften_theta_constraint else None,
    slack_penalty=slack_penalty,
)
print("solver success:", result.success)
print("solver status:", result.status)
print("objective:", result.objective)
print("first applied input:", result.u0)
print("first predicted state after applying u0:", result.X[1])
if result.slack.size:
    print("initial theta slack:", result.slack[0, 0])
else:
    print("no slack variable used in this solve")


## Run the Full Application Study

In [ ]:
steps = int(os.environ.get("THIMPC_CH4_PROJECT1_STEPS", "80"))
metrics = run_project1(steps=steps, output_dir=output_root / "ch4_project1")
metrics
